## Использование Gamac на реальных данных

### Импортируем нужные библиотеки

In [3]:
import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

In [3]:
# Импортируем датафрейм
data = pd.read_csv('../data/synthetic_logs_1kk.csv')

In [5]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=100)

dataset, best_model = gamac.run(table=data, target_measures=["OS"])

In [ ]:
# Получение топ-50 меток по лучшему классификатору
print(best_model.model.predict(dataset)[:50])

### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры. 

In [10]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=587x621>,
 <PIL.Image.Image image mode=RGBA size=595x842>,
 <PIL.Image.Image image mode=RGBA size=695x244>,
 <PIL.Image.Image image mode=RGBA size=500x500>,
 <PIL.Image.Image image mode=RGBA size=709x425>]

### Запустим подбор (в ячейке принты скрыты)

In [20]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

dataset, best_model = gamac.run(image=images, text=empty_texts, target_measures=["SYM"])

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Detected measures: [<Internal.SYM: ('sym', <function sym at 0x7e3ad9771a80>)>]
=== MEASURES 0.07890057563781738s, {'SYM': 0.0005099846384004896} ===
=== MEASURES 0.028857946395874023s, {'SYM': 0.040758537060877426} ===
=== ALGO 0.19092392921447754s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 11, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 0.5721681118011475s, FailedRun, {'bandwidth': 0.39883263382690765, 'max_iter': 61, 'tol': 8.269679095216381e-05} ===
=== MEASURES 0.07690000534057617s, {'SYM': 0.0} ===
=== ALGO 0.25650930404663086s, SuccessRun, {'name': 'DBSCAN', 'eps': 0.9349666245441267, 'eps_sq': 0.874162589011438, 'min_samples': 10} ===
=== ALGO 0.2384192943572998s, FailedRun, {'min_cluster_size': 8, 'min_samples': 10} ===
=== MEASURES 0.03168773651123047s, {'SYM': 0.04059243303598041} ===
=== ALGO 1.2476091384887695s, SuccessRun, {'threshold': 0.14343369565816808, 'branching_factor': 66, 'n_clusters': 12} ===
=== MEASURES 0.04053068161010742s, {'

In [21]:
print(best_model.model.predict(dataset))

[ 1  1  3  9  6  3  9  1  9  4  4  4  1  4  1  3  4  5  5  8  4  9  3  3
  5  8 10  7 10 10 10  6  1  6  6  4  8 10 10 10  5  1  7  1  4  3  1  1
  1  1  1  1  2  3  1  1  1  1  4 10  6  4 10  9  7 10 10  9  9  9 10  6
  4  1  4  4  4  9  9  9  1  4  6  6  4  1 10  6  7 10  1  3  1  6  4  1
  6 10  3 10  4  3  6  3  3  5  5  5  5  5 10 10  9  2  2  6  1  4  4  4
  5  3  1  9  3  3  1  1  1  1 10 10 10 10 10 10 10 10  1  7  7  7  7  7
  7  7  7  7  7  7  7  7  7  7  4  4  4  3  3  3  4 10 10 10  5  4  4  4
  4  4 10  4  4 10 10 10  8  4 10  4 10 10 10  0  8 10  1  1  1  1  1  1
 10 10 10 10 10  3  7  5  3 10  4  8 10  6  6  7  6  4  3  3  5  1 10 10
 10  5  4  4  4  1  4  1  7  7  7  4  4  5  5  5  1 10  1  4  1  8  4  4
  0  4 10  1  3  5  3  5  4  4  4  4  4 10  1 10  9  9  9  1  1  4  4  4
 10 10  6  6  6  6  9  9  4  4  4  3  8  3  3  9  3 10 10  2  2  2  3 10
 10 10 10  4  9  9  4  4  4  4  4  3  3 10 10  4  4  4  7  4  4  4 10 10
 10 10 10 10 10  4 10  7  1  3  3  4  8 10  7 10  7